# GoEmotions Sample with Derived Sentiment

How do the emotion annotations on 250 GoEmotions comments translate into positive, negative, or neutral sentiment?

Run the cells in order using the repository's `.venv` Python kernel with pandas installed. Internet access is required to read the official files. This notebook does not install packages or run a sentiment model.

[GoEmotions](https://github.com/google-research/google-research/tree/master/goemotions) contains Reddit comments annotated with 27 emotions plus neutral. A comment can have multiple emotion labels. This notebook samples the official agreement-filtered **training split** only; it does not combine training and evaluation data.

Sources: [dataset README](https://github.com/google-research/google-research/blob/master/goemotions/README.md), [ordered emotion labels](https://github.com/google-research/google-research/blob/master/goemotions/data/emotions.txt), and [sentiment mapping](https://github.com/google-research/google-research/blob/master/goemotions/data/sentiment_mapping.json). All downloads below use the same pinned revision.

The `sentiment` column is a **derived label**, not an original human sentiment annotation. Reddit comments may contain offensive language.

## Configure the sample

Use seed `42` to draw 250 comments without replacement. Change `SAMPLE_SIZE` or `RANDOM_SEED` to explore a different sample.

In [1]:
import csv
import json
from io import StringIO
from urllib.request import urlopen

import pandas as pd
from IPython.display import display

REVISION = "7951440944924ac61b6e2f9a2a3c715834005c40"
BASE_URL = (
    "https://raw.githubusercontent.com/google-research/google-research/"
    f"{REVISION}/goemotions/data"
)
SAMPLE_SIZE = 250
RANDOM_SEED = 42


def read_source(filename: str) -> str:
    with urlopen(f"{BASE_URL}/{filename}", timeout=60) as response:
        return response.read().decode("utf-8")

## Load the official training data and mappings

The headerless TSV contains comment text, comma-separated emotion IDs, and a comment ID. Emotion IDs are zero-based positions in `emotions.txt`. Disable CSV quote interpretation to preserve quotation marks in Reddit text.

In [2]:
emotion_names = read_source("emotions.txt").splitlines()
sentiment_mapping = json.loads(read_source("sentiment_mapping.json"))
train_data = pd.read_csv(
    StringIO(read_source("train.tsv")),
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"],
    dtype=str,
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
)

emotion_to_sentiment = {
    emotion: sentiment
    for sentiment, emotions in sentiment_mapping.items()
    for emotion in emotions
}
emotion_to_sentiment["neutral"] = "neutral"
assert set(emotion_to_sentiment) == set(emotion_names), "Emotion mapping mismatch"

print(f"Source revision: {REVISION}")
print(f"Training rows: {len(train_data):,}; emotion labels: {len(emotion_names)}")
display(train_data.dtypes.rename("dtype").to_frame())
display(train_data.eq("").sum().rename("empty_fields").to_frame())

Source revision: 7951440944924ac61b6e2f9a2a3c715834005c40
Training rows: 43,410; emotion labels: 28


,dtype
text,str
emotion_ids,str
id,str


,empty_fields
text,0
emotion_ids,0
id,0


## Sample comments and decode every emotion

Retain the original IDs and all emotion labels, including multi-label comments. The sample is random, not balanced by emotion or sentiment.

In [3]:
sample_data = train_data.sample(
    n=SAMPLE_SIZE, random_state=RANDOM_SEED, replace=False
).reset_index(drop=True)
sample_data["emotion_ids"] = sample_data["emotion_ids"].map(
    lambda value: [int(label_id) for label_id in value.split(",")]
)
assert all(
    0 <= label_id < len(emotion_names)
    for label_ids in sample_data["emotion_ids"]
    for label_id in label_ids
), "Emotion ID outside the official label list"
sample_data["emotion_labels"] = sample_data["emotion_ids"].map(
    lambda label_ids: [emotion_names[label_id] for label_id in label_ids]
)

## Derive three-class sentiment

The official JSON groups emotions into `positive`, `negative`, and `ambiguous`; it does not include the separate `neutral` emotion.

Use the agreed **unanimous** rule:

* `positive`: every emotion maps to positive
* `negative`: every emotion maps to negative
* `neutral`: any other combination, including ambiguous emotions, explicit neutral, or mixed groups

For example, joy + love is positive; anger + sadness is negative; joy + anger or joy + curiosity is neutral. Unknown or empty emotion labels raise an error instead of silently receiving a sentiment.

Collapsing ambiguous and mixed emotions into neutral is a project-specific choice, not a Google-provided sentiment annotation.

In [4]:
def derive_sentiment(emotions: list[str], mapping: dict[str, str]) -> str:
    if not emotions:
        raise ValueError("At least one emotion label is required")
    unknown = set(emotions) - mapping.keys()
    if unknown:
        raise ValueError(f"Unknown emotion labels: {sorted(unknown)}")
    groups = {mapping[emotion] for emotion in emotions}
    if groups == {"positive"}:
        return "positive"
    if groups == {"negative"}:
        return "negative"
    return "neutral"


sample_data["sentiment"] = sample_data["emotion_labels"].map(
    lambda emotions: derive_sentiment(emotions, emotion_to_sentiment)
)
sample_data = sample_data[["id", "text", "emotion_ids", "emotion_labels", "sentiment"]]

## Validate the conversion and sample

Check the agreed multi-emotion behavior, invalid inputs, sample size, decoded labels, and reproducible selection before displaying the data.

In [5]:
sentiment_cases = [
    (["joy"], "positive"),
    (["joy", "love"], "positive"),
    (["anger"], "negative"),
    (["anger", "sadness"], "negative"),
    (["neutral"], "neutral"),
    (["curiosity"], "neutral"),
    (["joy", "anger"], "neutral"),
    (["joy", "curiosity"], "neutral"),
    (["anger", "confusion"], "neutral"),
    (["joy", "neutral"], "neutral"),
]
for emotions, expected in sentiment_cases:
    assert derive_sentiment(emotions, emotion_to_sentiment) == expected, emotions
for invalid in ([], ["unknown_emotion"]):
    try:
        derive_sentiment(invalid, emotion_to_sentiment)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected invalid labels to fail: {invalid}")

assert len(sample_data) == SAMPLE_SIZE
assert sample_data["id"].is_unique
assert not sample_data.isna().any().any()
assert sample_data["text"].str.strip().ne("").all()
assert sample_data["sentiment"].isin(["positive", "negative", "neutral"]).all()
assert all(
    labels == [emotion_names[label_id] for label_id in label_ids]
    for label_ids, labels in zip(sample_data["emotion_ids"], sample_data["emotion_labels"])
)
assert sample_data["id"].tolist() == train_data.sample(
    n=SAMPLE_SIZE, random_state=RANDOM_SEED, replace=False
)["id"].tolist()
print(f"Validation passed for {len(sample_data)} rows and all sentiment edge cases.")

Validation passed for 250 rows and all sentiment edge cases.


## Inspect all 250 sampled comments

The table includes each comment's original emotion IDs, readable emotion labels, and derived sentiment. All sample rows and full comment text are displayed; the full training split is not printed.

In [6]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(sample_data)

print("Derived sentiment counts in this sample:")
display(
    sample_data["sentiment"].value_counts()
    .reindex(["positive", "negative", "neutral"], fill_value=0)
    .rename_axis("sentiment").to_frame("count")
)

,id,text,emotion_ids,emotion_labels,sentiment
0,edupnyh,The only way this works is if [NAME] is doing the Bird Box challenge and cant see the terms of the contract.,[27],[neutral],neutral
1,ediy7lp,Access should be hindered it's getting destroyed.,[3],[annoyance],negative
2,edv791a,Totally fair. All I was trying to remind everyone is don't fall in love with your crushes.,[4],[approval],positive
3,ed8lzbw,"I'm poly and jn the Raleigh area too, moved here for my wife's job, surprised to find so many people mention it haha.",[26],[surprise],neutral
4,edlvvfv,Naw man Asain men have an easier time. Most of my asain friend are pretty boy. Fuckbois lol,[1],[amusement],positive
5,eej3kcn,That scene traumatized me the first time I watched it,[14],[fear],negative
6,edl0m39,"No, some of us thought it was gross back then too.",[27],[neutral],neutral
7,efgc53r,Sounds like they’re not responsible and shouldn’t have the right to have these.,"[10, 22]","[disapproval, realization]",neutral
8,edyk56n,It’s awful. I couldn’t even see the ending at all.,"[11, 25]","[disgust, sadness]",negative
9,eeu4dpd,>West Covina That just happens to be where [NAME] lives!,[27],[neutral],neutral


Derived sentiment counts in this sample:


,count
sentiment,
positive,79
negative,57
neutral,114


## Interpretation and next steps

The counts describe this 250-row random training sample, not the full dataset or a balanced sentiment benchmark. Inspect `emotion_labels` alongside `sentiment` to distinguish explicit neutral annotations from ambiguous or conflicting emotions.

This mapping ignores the comment text when deriving sentiment, so it cannot independently verify sentiment or resolve sarcasm. Before using these labels to evaluate a sentiment model, decide whether the broad neutral bucket matches that evaluation's definition and use a held-out split rather than this training sample.

GoEmotions is based on English Reddit comments and has annotation and demographic biases; see the official README for limitations. To explore another sample, change the configuration and rerun all cells.